# **IV. Bài tập về nhà**

# **Bài 1. Cài đặt thuật giải AKT vào bài toán 15 puzzle(n>4).**

In [28]:
import copy
from heapq import heappush, heappop
n=5
rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

class priorityQueue:
  def __init__(self):
    self.heap = []
  def push(self, key):
    heappush(self.heap, key)
  def pop(self):
    return heappop(self.heap)
  def empty(self):
    if not self.heap:
      return True
    else:
      return False

class nodes:
  def __init__(self, parent, mats, empty_tile_posi, costs, levels):
    self.parent = parent
    self.mats = mats
    self.empty_tile_posi = empty_tile_posi
    self.costs = costs
    self.levels = levels
  def __lt__(self, nxt):
    return self.costs < nxt.costs
def calculateCosts(mats, final) -> int:
  count = 0
  for i in range(n):
    for j in range(n):
      if (mats[i][j]) and (mats[i][j] != final[i][j]):
        count += 1
  return count
def newNodes(mats, empty_tile_posi, new_empty_tile_posi, levels, parent, final) -> nodes:
  new_mats = copy.deepcopy(mats)
  x1 = empty_tile_posi[0]
  y1 = empty_tile_posi[1]
  x2 = new_empty_tile_posi[0]
  y2 = new_empty_tile_posi[1]
  new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]
  costs = calculateCosts(new_mats, final)
  new_nodes = nodes(parent, new_mats, new_empty_tile_posi, costs, levels)
  return new_nodes
def printMatrix(mats):
  for i in range(n):
    for j in range(n):
      print("%d " % (mats[i][j]), end = " ")
    print()
def isSafe(x, y):
  return x >= 0 and x < n and y >= 0 and y < n
def printPath(root):
  if root == None:
    return
  printPath(root.parent)
  printMatrix(root.mats)
  print()
def solve(initial, empty_tile_posi, final):
  pq = priorityQueue()
  costs = calculateCosts(initial, final)
  root = nodes(None, initial, empty_tile_posi, costs, 0)
  pq.push(root)
  while not pq.empty():
    minimum = pq.pop()
    if minimum.costs == 0:
      printPath(minimum)
      return
    for i in range(4):
      new_tile_posi = [
          minimum.empty_tile_posi[0] + rows[i],
          minimum.empty_tile_posi[1] + cols[i]
      ]
      if isSafe(new_tile_posi[0], new_tile_posi[1]):
        child = newNodes(minimum.mats, minimum.empty_tile_posi, new_tile_posi, minimum.levels + 1, minimum, final)
        pq.push(child)
initial = [
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18, 0, 19],
    [21, 22, 23, 24, 20]
]

final = [
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18, 19, 20],
    [21, 22, 23, 24, 0]
]

empty_tile_posi = [3, 3]
solve(initial, empty_tile_posi, final)

1  2  3  4  5  
6  7  8  9  10  
11  12  13  14  15  
16  17  18  0  19  
21  22  23  24  20  

1  2  3  4  5  
6  7  8  9  10  
11  12  13  14  15  
16  17  18  19  0  
21  22  23  24  20  

1  2  3  4  5  
6  7  8  9  10  
11  12  13  14  15  
16  17  18  19  20  
21  22  23  24  0  



# **Bài 2. Cài đặt bài toán người giao hàng theo thuật giải A*.**

In [41]:
import heapq
import itertools

def get_distance_matrix(coords):
    import math
    n = len(coords)
    dist = [[0]*n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            dist[i][j] = math.dist(coords[i], coords[j])
    return dist

def mst_cost(nodes, dist):
    if not nodes:
        return 0

    visited = set()
    start = next(iter(nodes))
    visited.add(start)
    edges = []

    total = 0
    while len(visited) < len(nodes):
        min_edge = float('inf')
        next_node = None

        for u in visited:
            for v in nodes:
                if v not in visited:
                    if dist[u][v] < min_edge:
                        min_edge = dist[u][v]
                        next_node = v

        visited.add(next_node)
        total += min_edge

    return total

def heuristic(current, unvisited, dist, start):
    if not unvisited:
        return dist[current][start]

    mst = mst_cost(unvisited, dist)

    min_to_mst = min(dist[current][u] for u in unvisited)
    min_from_mst = min(dist[u][start] for u in unvisited)

    return mst + min_to_mst + min_from_mst

def a_star_tsp(dist, start=0):
    n = len(dist)
    pq = []

    heapq.heappush(pq, (0, 0, start, frozenset([start]), [start]))

    while pq:
        f, g, current, visited, path = heapq.heappop(pq)

        if len(visited) == n:
            return path + [start], g + dist[current][start]

        unvisited = set(range(n)) - visited

        for nxt in unvisited:
            new_g = g + dist[current][nxt]
            new_visited = visited | {nxt}
            h = heuristic(nxt, unvisited - {nxt}, dist, start)

            heapq.heappush(
                pq,
                (new_g + h, new_g, nxt, new_visited, path + [nxt])
            )

    return None

real_path = [coords[i] for i in path]

total = 0
for i in range(len(path)-1):
    u = path[i]
    v = path[i+1]
    d = dist[u][v]
    total += d
    print(f"{coords[u]} → {coords[v]} = {d:.2f}")

print("Total cost:", round(total, 2))

(0, 0) → (1, 3) = 3.16
(1, 3) → (4, 3) = 3.00
(4, 3) → (6, 1) = 2.83
(6, 1) → (0, 0) = 6.08
Total cost: 15.07
